In [ ]:
# importing necessary libraries
import pandas as pd

In [ ]:
# Load the dataset
df = pd.read_csv("../data/DSI_kickstarterscrape_dataset.csv", encoding="latin1")

,project id,name,url,category,subcategory,location,status,goal,pledged,funded percentage,backers,funded date,levels,reward levels,updates,comments,duration
0,39409,WHILE THE TREES SLEEP,http://www.kickstarter.com/projects/emiliesaba...,Film & Video,Short Film,"Columbia, MO",successful,10500.0,11545.0,1.099524,66,"Fri, 19 Aug 2011 19:28:17 -0000",7,"$25,$50,$100,$250,$500,$1,000,$2,500",10,2,30.00
1,126581,Educational Online Trading Card Game,http://www.kickstarter.com/projects/972789543/...,Games,Board & Card Games,"Maplewood, NJ",failed,4000.0,20.0,0.005000,2,"Mon, 02 Aug 2010 03:59:00 -0000",5,"$1,$5,$10,$25,$50",6,0,47.18
2,138119,STRUM,http://www.kickstarter.com/projects/185476022/...,Film & Video,Animation,"Los Angeles, CA",live,20000.0,56.0,0.002800,3,"Fri, 08 Jun 2012 00:00:31 -0000",10,"$1,$10,$25,$40,$50,$100,$250,$1,000,$1,337,$9,001",1,0,28.00
3,237090,GETTING OVER - One son's search to finally kno...,http://www.kickstarter.com/projects/charnick/g...,Film & Video,Documentary,"Los Angeles, CA",successful,6000.0,6535.0,1.089167,100,"Sun, 08 Apr 2012 02:14:00 -0000",13,"$1,$10,$25,$30,$50,$75,$85,$100,$110,$250,$500...",4,0,32.22
4,246101,The Launch of FlyeGrlRoyalty &quot;The New Nam...,http://www.kickstarter.com/projects/flyegrlroy...,Fashion,Fashion,"Novi, MI",failed,3500.0,0.0,0.000000,0,"Wed, 01 Jun 2011 15:25:39 -0000",6,"$10,$25,$50,$100,$150,$250",2,0,30.00


In [ ]:
# Get the first 5 rows of the dataset
df.head()

In [ ]:
# Get the dimension of the dataset
df.shape

(45957, 17)

In [ ]:
# Get the summary of the dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45957 entries, 0 to 45956
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   project id         45957 non-null  int64  
 1   name               45957 non-null  str    
 2   url                45957 non-null  str    
 3   category           45957 non-null  str    
 4   subcategory        45957 non-null  str    
 5   location           44635 non-null  str    
 6   status             45957 non-null  str    
 7   goal               45957 non-null  float64
 8   pledged            45945 non-null  float64
 9   funded percentage  45957 non-null  float64
 10  backers            45957 non-null  int64  
 11  funded date        45957 non-null  str    
 12  levels             45957 non-null  int64  
 13  reward levels      45898 non-null  str    
 14  updates            45957 non-null  int64  
 15  comments           45957 non-null  int64  
 16  duration           45957 non-null

In [ ]:
# Check for missing values in the dataset
missing_count = df.isnull().sum()
missing_count[missing_count > 0]

location         1322
pledged            12
reward levels      59
dtype: int64

In [ ]:
# Calculate the percentage of missing values for each column to identify which columns have a significant amount of missing data
missing_percentage = (missing_count / len(df)) * 100
missing_percentage[missing_percentage > 0].round(2)

location         2.88
pledged          0.03
reward levels    0.13
dtype: float64

In [ ]:
# Check the first 10 rows with missing values in the "location" column
df[df["location"].isnull()].head(10)

status
failed        692
successful    630
Name: count, dtype: int64

In [ ]:
# Verify that the status of the campaigns with missing locations is either successful or failed
df[df["location"].isnull()]["status"].value_counts()

Location has 1,322 missing values (2.88%). The affected records include both successful and failed campaigns and still contain useful campaign data. Instead of dropping these rows, missing locations will be labeled as "Unknown" to preserve the records without assuming a location.

In [ ]:
# Fill missing locations with "Unknown"
df["location"] = df["location"].fillna("Unknown")

In [17]:
# Verify there are no missing locations left
df["location"].isnull().sum()

np.int64(0)

In [ ]:
# Inspect missing pledged values to see whether related campaign fields reveal a pattern
df[df["pledged"].isnull()][["name", "goal", "pledged", "funded percentage", "status"]]

,name,goal,pledged,funded percentage,status
1187,Xenonauts,50000.0,NaN,2.219487,live
4502,Twokinds Book Printing Drive,25000.0,NaN,6.303783,live
13381,HICKIES - TURN YOUR KICKS INTO SLIP-ONS,25000.0,NaN,4.907843,live
13802,Genie - Motion control time lapse device,150000.0,NaN,2.999618,live
25239,B9Creator - A High Resolution 3D Printer,50000.0,NaN,4.761380,live
29412,Phil Tippett's &quot;MAD GOD&quot;,40000.0,NaN,2.532484,live
31164,gTar: The First Guitar That Anybody Can Play,100000.0,NaN,2.846420,live
34274,Space Command,75000.0,NaN,1.796187,live
35032,BronyCon: The Documentary,60000.0,NaN,3.148185,live
40759,Two Guys SpaceVenture - by the creators of Spa...,500000.0,NaN,0.583738,live


### Pledged

There are 12 missing pledged values (0.03% of the dataset).
All 12 occur in campaigns with `status = "live"`.

Because Task #2 will determine whether live campaigns are retained,
these values are left unchanged for now.

If live campaigns remain in the dataset, pledged can be reconstructed as:

pledged = goal × funded percentage

In [20]:
# Check whether funded percentage is approximately pledged / goal
df[["goal", "pledged", "funded percentage"]].dropna().head(10)

,goal,pledged,funded percentage
0,10500.0,11545.0,1.099524
1,4000.0,20.0,0.005000
2,20000.0,56.0,0.002800
3,6000.0,6535.0,1.089167
4,3500.0,0.0,0.000000
5,3500.0,3582.0,1.023331
6,1000.0,280.0,0.280000
7,2000.0,2180.0,1.090000
8,1000.0,1125.0,1.125000
9,7500.0,9836.0,1.311527


In [ ]:
# Inspect missing reward levels values to see whether related campaign fields reveal a pattern
df[df["reward levels"].isnull()][["name", "levels", "reward levels", "status"]]

,name,levels,reward levels,status
78,DEAR HARVEY (Stories of Harvey Milk)...Educati...,0,NaN,failed
830,Having homeless create public art,0,NaN,failed
2720,The FullCircle Project: Skiing and Service in ...,0,NaN,successful
3067,Gifts and Crafts made from wood(Golf Cart),0,NaN,failed
3889,The Pyrosphere: large scale computer controlle...,0,NaN,failed
5129,Reanimation Library Coffer Builder,0,NaN,successful
8591,ENUR REQUIEM is going back into the studio!!!!!,0,NaN,failed
10624,Back &quot;The Pack&quot;!!!,0,NaN,successful
10712,Ballard High School Debate,0,NaN,successful
10749,THE FUTURE,0,NaN,failed


### Reward Levels

There are 59 missing values in `reward levels` (0.13% of the dataset).

All 59 missing values occur where `levels = 0`, which indicates these campaigns have no reward tiers rather than unknown reward information.

Therefore, the missing values will be replaced with `"No reward levels"`.

In [ ]:
# Fill missing reward levels with "No reward levels"
df["reward levels"] = df["reward levels"].fillna("No reward levels")

In [ ]:
# Verify there are no missing reward levels left
df["reward levels"].isnull().sum()

np.int64(0)